

# Statistical Analyses:


1. Survival analysis: Time-to-event modeling for NT→NZ conversion. Cox proportional hazards to identify what predicts faster conversion.

2. Markov chains: Model state transitions (No commitment → NT:C → NT:T → NT+NZ). Calculate transition probabilities, steady-state distributions, expected time in each state.

3. Logistic regression: Predict which companies will convert NT→NZ based on cohort year, sector, region, initial status type.

4. Clustering: Group companies by their trajectory patterns (fast adopters, slow movers, dropouts, leapfroggers).

5. Churn analysis: Model why companies lose commitments (those 109 NZ losses in 2024→2025).

#### Temporal Analyses

6. Acceleration metrics: Is adoption speeding up? Compare slopes between cohorts.

7. Momentum indicators: Leading vs lagging sectors/regions in adoption waves.

8. Seasonality: Do commitments cluster around specific times (COP meetings, reporting cycles)?

#### Network/Portfolio Analyses

9. Portfolio risk: If X% typically drop targets, what's the expected stable state?


10. Contagion effects: If you had company relationships, model peer influence on adoption.

11. Optimal pathway: Which progression sequence has highest retention? (Direct to both vs stepwise)

#### Predictive Models

12. Time series forecasting: Project 2026-2030 adoption rates using ARIMA or exponential smoothing.

13. Cohort retention curves: Kaplan-Meier style plots showing retention by entry year.

14. Propensity scoring: Given attributes, probability of NT→NZ within 1/2/3 years.

#### What Would Be Most Insightful?

Given data quality, I'd prioritize:
- Markov chain model (clean state transitions, interpretable probabilities)
- Survival analysis (directly answers "when will they convert")
- Churn analysis (explains the anomalous NZ losses)





## What correlations to test:

1. Carbon credit usage vs commitment types
   - Companies saying they'll use carbon credits → higher likelihood of having CN/NZ/SBT?
   - Does CC usage correlate with faster NT→NZ conversion?

2. Commitment co-occurrence
   - If you have SBT, how likely to also have NZ?
   - If you have CN, how likely to mention CC usage?
   - Which commitments cluster together?

3. Temporal patterns
   - Does CC usage percentage change over time?
   - Does CC acceptance correlate with cohort year?

4. Regional/sectoral patterns
   - Which regions/sectors more likely to use CCs?
   - Does this correlate with commitment types?

## Statistical tests:

- Chi-square test: Independence between categorical variables (CC yes/no × SBT yes/no)
- Cramér's V: Strength of association (0-1 scale)
- Phi coefficient: For 2×2 tables specifically
- Point-biserial correlation: Binary (CC yes/no) vs continuous (number of commitments)
- Tetrachoric correlation: Underlying continuous relationship between two binary variables

## Example output:

"Companies using carbon credits are 2.3x more likely to have NZ targets (χ²=45.3, p<0.001, Cramér's V=0.28)"


## Markov 


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_excel('historic_new.xlsx', sheet_name='sbti evolution ')
df.columns = df.columns.str.strip()
df['company'] = df['company'].astype(str)

years = ['2021', '2022', '2023', '2024', '2025']

# Define states
def get_state(nt, nz):
    nt = str(nt) if pd.notna(nt) else 'None'
    nz = str(nz) if pd.notna(nz) else 'None'
    
    if nt == 'None' and nz == 'None':
        return 'No commitment'
    if nt != 'None' and nz == 'None':
        return f'NT:{nt}'
    if nt == 'None' and nz != 'None':
        return f'NZ:{nz}'
    return f'NT:{nt}+NZ:{nz}'

# Build states for each company-year
states = {}
for _, row in df.iterrows():
    company = row['company']
    states[company] = {}
    for year in years:
        nt = row[f'{year}_NT_Status']
        nz = row[f'{year}_NZ_Status']
        states[company][year] = get_state(nt, nz)

# Count transitions
transitions = {}
for company in states:
    for i in range(len(years) - 1):
        from_state = states[company][years[i]]
        to_state = states[company][years[i+1]]
        
        if from_state not in transitions:
            transitions[from_state] = {}
        if to_state not in transitions[from_state]:
            transitions[from_state][to_state] = 0
        
        transitions[from_state][to_state] += 1

# Get all unique states
all_states = sorted(set(s for company in states.values() for s in company.values()))

# Build transition matrix
n = len(all_states)
matrix = np.zeros((n, n))
state_to_idx = {s: i for i, s in enumerate(all_states)}

for from_state in transitions:
    from_idx = state_to_idx[from_state]
    row_sum = sum(transitions[from_state].values())
    
    for to_state, count in transitions[from_state].items():
        to_idx = state_to_idx[to_state]
        matrix[from_idx, to_idx] = count / row_sum

# Print transition matrix
print("TRANSITION PROBABILITY MATRIX")
print("Rows = current state, Columns = next state\n")

# Print header
print(f"{'From State':<20}", end="")
for state in all_states:
    print(f"{state:<20}", end="")
print()

# Print matrix
for i, from_state in enumerate(all_states):
    print(f"{from_state:<20}", end="")
    for j in range(n):
        if matrix[i, j] > 0:
            print(f"{matrix[i, j]:.3f}              ", end="")
        else:
            print(f"{'.':<20}", end="")
    print()

# Steady state (eigenvector for eigenvalue 1)
eigenvalues, eigenvectors = np.linalg.eig(matrix.T)
steady_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
steady = np.real(eigenvectors[:, steady_idx])
steady = steady / steady.sum()

print("\n\nSTEADY STATE DISTRIBUTION")
print("Long-run equilibrium probabilities:\n")
for i, state in enumerate(all_states):
    print(f"{state:<30} {steady[i]:.3f} ({steady[i]*100:.1f}%)")

# Key transitions
print("\n\nKEY TRANSITIONS (>5% probability)")
key = []
for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.05 and i != j:
            key.append((from_state, to_state, matrix[i, j]))

key.sort(key=lambda x: -x[2])
for from_s, to_s, prob in key:
    print(f"{from_s:<25} → {to_s:<25} {prob:.3f}")

# Visualization: Sankey of top transitions
sources = []
targets = []
values = []
labels = all_states.copy()

for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.02:
            sources.append(i)
            targets.append(j)
            values.append(matrix[i, j])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        label=labels,
        color='lightblue'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title="Markov Chain Transition Probabilities (edges > 2%)",
    height=800,
    width=1200
)

import os
os.chdir('/mnt/user-data/outputs')
fig.write_html('markov_transitions.html')

# Save transition matrix
tm_df = pd.DataFrame(matrix, index=all_states, columns=all_states)
tm_df.to_csv('transition_matrix.csv')

# Save steady state
ss_df = pd.DataFrame({'state': all_states, 'probability': steady})
ss_df.to_csv('steady_state.csv', index=False)

print("\n\nFiles: markov_transitions.html, transition_matrix.csv, steady_state.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'historic_new.xlsx'

## Cross-sectional analyses (2025 only):

Chi-square independence tests: CC usage vs NZ, CC vs SBT, etc.
Cramér's V correlation matrix: Heatmap of all commitment associations
Conditional probabilities: P(NZ | SBT), P(CC | CN), etc.
Logistic regression: Predict NZ from sector, region, CC, NT status
Cluster analysis: Group companies by commitment profile
Sector/region benchmarking: Which sectors lead in each commitment type

## Longitudinal with 2024-2025:

Year-over-year changes: Who gained/lost each commitment
Transition analysis: 2024 state → 2025 state (9x9 matrix)
Upgrade/downgrade rates: C→T vs T→C vs dropouts
Logistic for change: Predict who converts/drops based on 2024 profile

# 2025 only stats


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1, nrows=500)
df.columns = ['company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        v = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: V={v:.3f}, p={p:.4f}")

print("\nCONDITIONAL PROB")
print(f"P(nz|has_nt) = {df.loc[df['has_nt']==1, 'nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df.loc[df['cc_yes']==1, 'nz'].mean():.3f}")
print(f"P(cc_yes|cn) = {df.loc[df['cn']==1, 'cc_yes'].mean():.3f}")
print(f"P(has_nt|re100) = {df.loc[df['re100']==1, 'has_nt'].mean():.3f}")

print("\nLOGISTIC: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100']]
lr = LogisticRegression(max_iter=1000)
lr.fit(X, df['nz'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")

print("\nCLUSTERS")
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[vars_test])
for i in range(4):
    n = (df['cluster']==i).sum()
    means = df[df['cluster']==i][vars_test].mean()
    print(f"C{i} (n={n}): nt={means['has_nt']:.2f}, nz={means['nz']:.2f}, cc={means['cc_yes']:.2f}, cn={means['cn']:.2f}, re={means['re100']:.2f}")

n = len(vars_test)
corr = np.zeros((n, n))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            corr[i,j] = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))

print("\nCRAMERS V")
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

fig = go.Figure(go.Heatmap(z=corr, x=vars_test, y=vars_test, colorscale='Blues', text=np.round(corr, 3), texttemplate='%{text}'))
fig.update_layout(title="Cramers V", height=600, width=700)
fig.write_html('cramers_v.html')




CHI-SQUARE
has_nt vs nz: V=0.309, p=0.0000
has_nt vs cc_yes: V=0.099, p=0.0265
has_nt vs cn: V=0.044, p=0.3234
has_nt vs re100: V=0.280, p=0.0000
nz vs cc_yes: V=0.435, p=0.0000
nz vs cn: V=0.472, p=0.0000
nz vs re100: V=0.284, p=0.0000
cc_yes vs cn: V=0.040, p=0.3748
cc_yes vs re100: V=0.150, p=0.0008
cn vs re100: V=0.065, p=0.1449

CONDITIONAL PROB
P(nz|has_nt) = 0.737
P(nz|cc_yes) = 0.751
P(cc_yes|cn) = 0.396
P(has_nt|re100) = 0.623

LOGISTIC: PREDICT NZ
has_nt: OR=4.71
cc_yes: OR=10.07
cn: OR=0.01
re100: OR=3.86

CLUSTERS
C0 (n=88): nt=0.60, nz=0.98, cc=0.00, cn=0.00, re=0.28
C1 (n=171): nt=0.40, nz=0.97, cc=1.00, cn=0.00, re=0.25
C2 (n=91): nt=0.26, nz=0.00, cc=0.40, cn=1.00, re=0.10
C3 (n=149): nt=0.07, nz=0.00, cc=0.09, cn=0.00, re=0.00

CRAMERS V
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280 

In [4]:
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280  0.284   0.150  0.065  1.000


## overlap matrix

In [5]:
# List of key variables
vars_summary = ['nz', 'cn', 're100', 'has_nt', 'cc_yes']

# Initialize empty DataFrame for the overlap counts
overlap_matrix = pd.DataFrame(index=vars_summary, columns=vars_summary)

# Fill the matrix
for row_var in vars_summary:
    for col_var in vars_summary:
        overlap_matrix.loc[row_var, col_var] = ((df[row_var]==1) & (df[col_var]==1)).sum()

# Convert to integer
overlap_matrix = overlap_matrix.astype(int)

print(overlap_matrix)


         nz  cn  re100  has_nt  cc_yes
nz      252   0     65     115     166
cn        0  91      9      24      36
re100    65   9     77      48      48
has_nt  115  24     48     156      81
cc_yes  166  36     48      81     221


## Further Analysis 2025-2024

In [ ]:
print("ANALYSIS 1: CHI-SQUARE INDEPENDENCE TESTS (2025)")


vars_2025 = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
chi_results = []

for i, var1 in enumerate(vars_2025):
    for var2 in vars_2025[i+1:]:
        ct = pd.crosstab(df[var1], df[var2])
        chi2, p, dof, exp = chi2_contingency(ct)
        n = ct.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        chi_results.append({
            'var1': var1,
            'var2': var2,
            'chi2': chi2,
            'p': p,
            'cramers_v': cramers_v
        })
        print(f"{var1} vs {var2}: chi2={chi2:.2f}, p={p:.4f}, V={cramers_v:.3f}")

chi_df = pd.DataFrame(chi_results)

In [ ]:

print("\nANALYSIS 2: CONDITIONAL PROBABILITIES (2025)")


conditions = [
    ('has_nz_2025', 'has_nt_2025'),
    ('has_nz_2025', 'cc_usage'),
    ('cc_usage', 'cn'),
    ('has_nt_2025', 're100')
]

for outcome, given in conditions:
    prob = df[df[given]==1][outcome].mean()
    print(f"P({outcome} | {given}=1) = {prob:.3f}")

In [ ]:


print("\nANALYSIS 3: YEAR-OVER-YEAR CHANGES")


for var in ['has_nt', 'has_nz']:
    gained = ((df[f'{var}_2024']==0) & (df[f'{var}_2025']==1)).sum()
    lost = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==0)).sum()
    kept = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==1)).sum()
    print(f"{var}: +{gained} gained, -{lost} lost, {kept} kept")


In [ ]:

print("\nANALYSIS 4: STATE TRANSITIONS 2024->2025")


def state(row, year):
    nt = row[f'{year}_NT_Status']
    nz = row[f'{year}_NZ_Status']
    if pd.isna(nt) and pd.isna(nz): return 'None'
    if pd.notna(nt) and pd.isna(nz): return 'NT'
    if pd.isna(nt) and pd.notna(nz): return 'NZ'
    return 'Both'

df['state_2024'] = df.apply(lambda r: state(r, '2024'), axis=1)
df['state_2025'] = df.apply(lambda r: state(r, '2025'), axis=1)

trans_ct = pd.crosstab(df['state_2024'], df['state_2025'])
print(trans_ct)



In [ ]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025")


X = df[['has_nt_2025', 'cc_usage', 'cn', 're100']].fillna(0)
y = df['has_nz_2025']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")




In [ ]:
print("\nANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)")


for group in ['sector', 'region']:
    print(f"\n{group}:")
    agg = df.groupby(group)[['has_nt_2025', 'has_nz_2025', 'cc_usage']].mean()
    print(agg.round(3))


In [ ]:
print("\nANALYSIS 7: CLUSTER ANALYSIS (2025)")


X_cluster = df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].fillna(0)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

for i in range(4):
    cluster_df = df[df['cluster']==i]
    n = len(cluster_df)
    profile = cluster_df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].mean()
    print(f"\nCluster {i} (n={n}):")
    print(profile.round(3).to_dict())


In [ ]:
print("\nANALYSIS 8: UPGRADE/DOWNGRADE RATES")


df['nt_upgrade'] = ((df['2024_NT_Status']=='C') & (df['2025_NT_Status']=='T')).astype(int)
df['nt_downgrade'] = ((df['2024_NT_Status']=='T') & (df['2025_NT_Status']=='C')).astype(int)
df['nz_upgrade'] = ((df['2024_NZ_Status']=='C') & (df['2025_NZ_Status']=='T')).astype(int)
df['nz_downgrade'] = ((df['2024_NZ_Status']=='T') & (df['2025_NZ_Status']=='C')).astype(int)

print(f"NT upgrades: {df['nt_upgrade'].sum()}")
print(f"NT downgrades: {df['nt_downgrade'].sum()}")
print(f"NZ upgrades: {df['nz_upgrade'].sum()}")
print(f"NZ downgrades: {df['nz_downgrade'].sum()}")

In [ ]:
print("\nANALYSIS 9: LOGISTIC FOR CHANGE - WHO CONVERTS 2024->2025")


converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==1)]
non_converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==0)]
subset = pd.concat([converters, non_converters])

X_change = subset[['has_nt_2024', 'cc_usage', 'cn', 're100']].fillna(0)
y_change = (subset['has_nz_2025']==1).astype(int)

if len(y_change.unique()) > 1:
    lr_change = LogisticRegression(max_iter=1000)
    lr_change.fit(X_change, y_change)
    
    print("Predictors of NZ adoption (among non-NZ in 2024):")
    for feat, coef in zip(X_change.columns, lr_change.coef_[0]):
        odds_ratio = np.exp(coef)
        print(f"{feat}: OR={odds_ratio:.3f}")


In [ ]:
print("\nANALYSIS 10: CRAMERS V CORRELATION MATRIX")


vars_corr = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
n_vars = len(vars_corr)
corr_matrix = np.zeros((n_vars, n_vars))

for i, v1 in enumerate(vars_corr):
    for j, v2 in enumerate(vars_corr):
        if i == j:
            corr_matrix[i, j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, p, dof, exp = chi2_contingency(ct)
            n = ct.sum().sum()
            cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
            corr_matrix[i, j] = cramers_v

corr_df = pd.DataFrame(corr_matrix, index=vars_corr, columns=vars_corr)
print(corr_df.round(3))

In [ ]:
# VISUALIZATIONS
fig1 = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    x=vars_corr,
    y=vars_corr,
    colorscale='Blues'
))
fig1.update_layout(title="Cramers V Correlation Matrix", height=600, width=700)

fig2 = go.Figure(data=[
    go.Bar(name='2024', x=['NT', 'NZ'], y=[df['has_nt_2024'].sum(), df['has_nz_2024'].sum()]),
    go.Bar(name='2025', x=['NT', 'NZ'], y=[df['has_nt_2025'].sum(), df['has_nz_2025'].sum()])
])
fig2.update_layout(title="Commitment Counts 2024 vs 2025", barmode='group')

fig3 = make_subplots(rows=1, cols=2, subplot_titles=['By Sector', 'By Region'])
sector_agg = df.groupby('sector')['has_nz_2025'].mean().sort_values()
region_agg = df.groupby('region')['has_nz_2025'].mean().sort_values()
fig3.add_trace(go.Bar(x=sector_agg.values, y=sector_agg.index, orientation='h'), row=1, col=1)
fig3.add_trace(go.Bar(x=region_agg.values, y=region_agg.index, orientation='h'), row=1, col=2)
fig3.update_layout(title="NZ Adoption Rate by Sector and Region", height=400, width=1000)

import os
os.chdir('/mnt/user-data/outputs')

fig1.write_html('cramers_v_matrix.html')
fig2.write_html('yoy_comparison.html')
fig3.write_html('sector_region_benchmark.html')

chi_df.to_csv('chi_square_tests.csv', index=False)
corr_df.to_csv('correlation_matrix.csv')
trans_ct.to_csv('state_transitions.csv')
df.to_csv('analysis_dataset.csv', index=False)

print("\nFiles saved")